# Tutorial: Palette catalog summaries

Audience:
- Developers and designers who want to analyze a CSV palette catalog using only the Python standard library.

Prerequisites:
- Basic Python (functions, dictionaries, lists).
- Familiarity with reading CSV files.

Learning goals:
- By the end, the reader can read a CSV catalog, normalize its fields, group rows by a key, and summarize each group.


## Outline

1. Setup
2. Reading and normalizing the catalog
3. Grouping palettes by primary color
4. Running the summary on the real catalog
5. Exercises


In [ ]:
# Setup cell: keep it short and deterministic
from __future__ import annotations

import csv
from collections import OrderedDict

CATALOG_PATH = "/inputs/colors.csv"
CATALOG_PATH


## Step 1 - Define the grouping function

`summarize_palettes(csv_path)` reads a UTF-8 CSV catalog using its header row.
For each row it strips surrounding whitespace from `Primary (Hex)` and `Product Type`,
uppercases the primary color value, and groups rows by that normalized primary color.

Each returned group is an object (a `dict`) with exactly three keys:

- `primary`: the normalized (stripped, uppercased) primary color text.
- `count`: the number of rows in the group, including duplicate rows.
- `product_types`: the sorted list of unique stripped product-type texts seen in the group.

Groups are ordered by `count` descending, then by `primary` ascending (normal Python
lexicographic sorting). A header-only input (no data rows) returns an empty list.


In [ ]:
def summarize_palettes(csv_path):
    """Group a palette catalog CSV by normalized primary color.

    Returns a list of dicts: {"primary": str, "count": int, "product_types": [str, ...]}
    ordered by count descending, then primary ascending.
    """
    groups = OrderedDict()

    with open(csv_path, newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            raw_primary = row.get("Primary (Hex)", "")
            raw_product_type = row.get("Product Type", "")

            primary = raw_primary.strip().upper()
            product_type = raw_product_type.strip()

            if primary not in groups:
                groups[primary] = {"count": 0, "product_types": set()}

            groups[primary]["count"] += 1
            groups[primary]["product_types"].add(product_type)

    summaries = [
        {
            "primary": primary,
            "count": data["count"],
            "product_types": sorted(data["product_types"]),
        }
        for primary, data in groups.items()
    ]

    summaries.sort(key=lambda item: (-item["count"], item["primary"]))
    return summaries

summarize_palettes


## Step 2 - Run the summary on the real catalog

The next cell applies `summarize_palettes` to the bundled catalog at `/inputs/colors.csv`
and stores the result in `palette_summary` so it can be inspected or reused.


In [ ]:
palette_summary = summarize_palettes("/inputs/colors.csv")
len(palette_summary), palette_summary[:3]


## Exercises

- Try passing a copy of the catalog with the columns reordered. Confirm the result is unchanged,
  since `summarize_palettes` reads columns by header name rather than by position.
- Build a small CSV string with duplicate rows and lowercase hex values (e.g. `#3b82f6`),
  write it to a temp file, and confirm duplicates increase `count` while the normalized
  `primary` is uppercase.
- Build a header-only CSV (no data rows) and confirm `summarize_palettes` returns `[]`.
- Add a row whose `Product Type` contains a comma inside quotes, for example
  `"Widgets, Inc."`, and confirm it survives as one product type rather than being split.
- Predict, before running, which primary color group will have the highest `count` in
  `palette_summary`, then check your prediction.
